<a href="https://colab.research.google.com/github/Rishikaj03/GenAI/blob/main/GenAIPractical8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from transformers import pipeline, AutoModelForSeq2SeqLM, AutoTokenizer, AutoModelForQuestionAnswering
import transformers
import pandas as pd
import torch

# 1. Define the input statement
text = "Renewable energy sources like solar and wind power are essential for combating climate change, reducing carbon footprints, and ensuring a sustainable future for global economies."
print("Input Text:")
print(text)
print("-" * 50)

# 2. Text Classification
classifier = pipeline("text-classification")
outputs = classifier(text)
print("Classification Output:")
display(pd.DataFrame(outputs))
print("-" * 50)

# 3. Text Summarization using T5
model_name = "t5-small"
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
tok = AutoTokenizer.from_pretrained(model_name)

input_text = "summarize: " + text
inputs = tok(input_text, return_tensors="pt", max_length=512, truncation=True)

summary_ids = model.generate(
    inputs["input_ids"],
    max_length=56,
    min_length=10,
    length_penalty=2.0,
    num_beams=4,
    early_stopping=True
)

summary = tok.decode(
    summary_ids[0],
    skip_special_tokens=True,
    clean_up_tokenization_spaces=True
)
print("Summarization Output:")
print({"summary_text": summary})
print("-" * 50)


[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


Input Text:
Renewable energy sources like solar and wind power are essential for combating climate change, reducing carbon footprints, and ensuring a sustainable future for global economies.
--------------------------------------------------


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

...

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

Summarization Output:
{'summary_text': 'renewable energy sources like solar and wind power are essential for combating climate change, reducing carbon footprints and ensuring a sustainable future for global economies.'}
--------------------------------------------------


In [2]:

# 4. Question Answering
qa_model_name = "distilbert-base-cased-distilled-squad"
qa_model = AutoModelForQuestionAnswering.from_pretrained(qa_model_name)
qa_tokenizer = AutoTokenizer.from_pretrained(qa_model_name)

question = "What are essential for combating climate change?"
qa_inputs = qa_tokenizer(question, text, return_tensors="pt")

with torch.no_grad():
    qa_outputs = qa_model(**qa_inputs)

answer_start = torch.argmax(qa_outputs.start_logits)
answer_end = torch.argmax(qa_outputs.end_logits) + 1

answer = qa_tokenizer.convert_tokens_to_string(
    qa_tokenizer.convert_ids_to_tokens(
        qa_inputs["input_ids"][0][answer_start:answer_end]
    )
)

print("Question Answering Output:")
print(f"Question: {question}")
print(f"Answer: {answer}")

config.json:   0%|          | 0.00/473 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  261MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

Question Answering Output:
Question: What are essential for combating climate change?
Answer: Renewable energy sources
